# 03 · 학습 — **대조군** (`acm`, Mamba 디코더 / carry off)

**프로토콜 (전 그룹 공통)** — 학습 **150k step · seed 4개 · lr 고정(sweep 없음)**, eval **150k 체크포인트 × 5회 반복**(rep 마다 env seed 변경) → 모델당 4×5 = 20 run.

`acm` = **`ours` 와 완전히 같은 백본**(Mamba-1 디코더)에서 **carry·BiMamba·MOSAIC 만 끈 것**.

즉 우리 기여를 전부 제거한 **바닥**. 논문 헤드라인이 "plain Mamba 디코더 대비 [XX]p 개선"이라
**이 수치가 그 주장의 분모**다. 없으면 개선폭을 못 쓴다.

1모델 × 4 seed = **4잡**.

⚠️ 옛 acm 런(lr 3e-5 등)을 가져다 쓰면 안 됨 — lr/step/seed 가 다르면 비교가 무효.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion' (aloha)
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
GPUS  = cf.v23.available_gpus()   # 이 노드에 실제 보이는 GPU
NGPU  = len(GPUS)                 # 4개면 4잡씩 청크로 (하드코딩 X)
TAGS  = cf.GROUP_ACM       # ['acm']

print('GPU  :', GPUS, f'({NGPU}개)')
print('학습:', TAGS, '| task:', TASK, '| seeds:', SEEDS)
print('steps:', f'{cf.STEPS:,}', '| 잡:', len(TAGS) * len(SEEDS))
for t in TAGS:
    pol, lr, K, extra, cp = cf.v23.MODEL_CONFIGS[t]
    print(f'  {t:<12} {pol:<34} lr={lr:<7} K={K:<4} pairs={cp}')

## 커맨드 확인 (dry-run)

In [ ]:
for t in TAGS:
    print(cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0))
    print()

## 학습 (resume 자동 — 끊겨도 다시 돌리면 이어감)

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, ngpu=NGPU)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)